# Taiga-tundra ecotone (TTE) map on MAAP DPS

Runs `lib/tte_map.py` (Python port of the GEE TTE workflow, Montesano et al. 2020 ERL) as the DPS
algorithm `run_tte_map`, one or more 1-degree tiles per job. Each job writes:

* one 5-band COG per tile (`tte_<tcc label>_<tile_num>.tif`: tte_domain, tte_zones, tte_abruptness,
  tte_classification, tcc), and
* STAC metadata (`write_stac=true`): `catalog.json`, `tte_map/collection.json` and one item per tile.
  DPS publishes the items in the MAAP DPS STAC catalog, collection `{user}__<deployed name>__{version}`
  (DPS adds a suffix to the deployed name, e.g. `montesano__run_tte_map_13__tte_map_v1.1`),
  where `notebooks/TTE_review_map.ipynb` maps them.

**What each STAC item records** (see `lib/tte_stac.py`):

| Group | Fields |
|---|---|
| Identity | id `tte_<tcc label>_<tile_num>_<parameter signature>`; footprint = the 1-degree tile; datetime = `tcc_year` (Jan 1) |
| How it was made | `tte:tcc_source`, `tte:tcc_label`, `tte:tcc_year`, `tte:tcc_options`, `tte:tcc_paths`, `tte:tcc_description`, `tte:water`, `tte:domain`, `tte:gee_pyramid_emulation`, `tte:zone_thresholds_pct`, `tte:code_version` (git tag), `tte:method`; DPS adds `maap-dps:*` (user, algorithm, job tag) and `created` |
| What is in the file | asset `tte` (COG), band names and descriptions, `raster:bands` (float32, nodata NaN, 30 m), `classification:classes` on band 4 (values 1-11, names, colors), `proj:*` (polar LAEA grid, WKT2) |
| Per-tile summary | `tte:domain_km2`, `tte:classified_km2`, `tte:classified_share_of_domain`, `tte:mean_tcc_domain`, `tte:mean_abruptness`, `tte:class_km2`, `tte:group_share` |
| Citation | `sci:doi`, `sci:citation` (Montesano et al. 2020); collection license CC-BY-4.0 |

Re-running a tile with the same parameters replaces its item; other parameters (tree cover source,
year, versions, options) give a new item in the same collection.

**Where to run:** MAAP Hub, a Python kernel; submitting needs your MAAP login, which the Hub kernel
has. Before registering, the code must be on GitHub with the tag named in `ALGO_VERSION`
(`github.com/pahbs/tte`).

**MAAP's OGC DPS (maap-py 5).** Algorithms are now OGC processes described by a CWL file; the
**Register Algorithm** form on MAAP Hub writes the CWL from `dps/alg_tte_map/algorithm_config.yml`. Inputs reach
`run.sh` by name (`--tile_num 4209 --tcc_source hansen ...`), and jobs are submitted with
`maap.submit_job(process_id, inputs, queue, tag=...)`.

In [ ]:
import os, sys, glob, json, datetime, warnings
warnings.filterwarnings('ignore', message='.*geographic CRS.*')
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from maap.maap import MAAP
maap = MAAP()

REPO = os.path.expanduser('~/code/tte')
sys.path.insert(0, f'{REPO}/lib')
import tte_tcc_sources as ts
ALG_YML = f'{REPO}/dps/alg_tte_map/algorithm_config.yml'   # load this file in the Register Algorithm form
ALGO_ID = 'run_tte_map'
ALGO_VERSION = 'tte_map_v1.1'        # git tag in github.com/pahbs/tte
USER = 'montesano'
QUEUE = 'maap-dps-worker-8gb'
import requests
# DPS names the deployed algorithm with a suffix (e.g. run_tte_map_13): its outputs go to
# dps_output/<deployed name>/<version>/<date>/... and its STAC items to {user}__<deployed name>__<version>
OUT_GLOB = os.path.expanduser(f'~/my-private-bucket/dps_output/{ALGO_ID}*/{ALGO_VERSION}')


def find_collection():
    # the DPS STAC collection of this algorithm version (None until the first job's items are published)
    r = requests.get('https://dps-stac.maap-project.org/collections', params={'limit': 1000}, timeout=60)
    ids = [c['id'] for c in r.json().get('collections', [])
           if c['id'].startswith(f'{USER}__{ALGO_ID}') and c['id'].endswith(f'__{ALGO_VERSION}')]
    return sorted(ids)[-1] if ids else None


COLLECTION = find_collection()
print('STAC collection:', COLLECTION or '(none yet: created when the first job is published)')

## 1. Register (once per version)

maap-py 5 has no `register_algorithm_from_yaml_file`. Open the **Register Algorithm** form on MAAP
Hub (MAAP menu or launcher), load `dps/alg_tte_map/algorithm_config.yml` (the OGC-format config:
`code_repository`, `algorithm_version` = the git tag, `base_container_url` + `build_command`,
`ram_min`/`cores_min`, named inputs), check the fields, and register. It builds the container
(running `build.sh`) and the CWL process. Wait for the build to finish before submitting.
Registering again with the same name and version replaces the algorithm. The cell below shows the
config as the form will read it.

In [ ]:
import yaml
cfg = yaml.safe_load(open(ALG_YML))
print(ALG_YML, '\n')
for k in ['algorithm_name', 'algorithm_version', 'code_repository', 'run_command', 'build_command',
          'base_container_url', 'ram_min', 'cores_min']:
    print(f'{k:20s} {cfg[k]}')
print('\ninputs (name / default):')
for i in cfg['inputs']:
    print(f"  {i['name']:16s} {i.get('default', '(required)')}")

After the build finishes: the OGC process id of the algorithm, needed to submit jobs.

In [ ]:
r = maap.list_algorithms(algorithm_name=ALGO_ID, algorithm_version=ALGO_VERSION)
listing = r.json() if hasattr(r, 'json') else r
procs = listing.get('processes', listing) if isinstance(listing, dict) else listing
print(json.dumps(procs, indent=1)[:1500])
# the OGC endpoint is addressed by the numeric processID (its link: /ogc/processes/<processID>), not by 'id'
PROCESS_ID = procs[0].get('processID', procs[0]['id']) if procs else None
print('PROCESS_ID:', PROCESS_ID)

## 2. Inputs: choose the tree cover

The algorithm takes any tree cover source registered in `lib/tte_tcc_sources.py` (printed below),
by name (`tcc_source`), year (`tcc_year`) and options (`tcc_options`, 'key=value;key=value'). Pick a
run in `RUNS` (add your own) and set `RUN`; nothing is chosen by default.

* `floor=5` works with any source: TCC below 5 % set to 0 (see the comparison notebooks).
* `path` reads your own percent cover rasters (`tcc_paths`, s3:// or https://) and needs `out_name`.
* `write_stac='true'` writes the STAC metadata (keep it on).

`IDENTIFIER` is the DPS job tag: it groups jobs in `dps_output` and is `maap-dps:tag` in STAC.

In [ ]:
print(ts.list_sources())

In [ ]:
COMMON = {
    'tile_num': '',
    'tcc_options': 'none',
    'tcc_paths': 'none',
    'out_name': 'none',
    'gee_emulation': 'true',
    'domain_path': 's3://maap-ops-workspace/shared/montesano/tte/static/tte_domain_v201902v2.vrt',
    'tiles_gpkg': 's3://maap-ops-workspace/shared/montesano/databank/tte_tiles.gpkg',
    'write_stac': 'true',
}
RUNS = {   # examples; add your own
    'hansen_2010_gee_version': dict(tcc_source='hansen', tcc_year='2010', tcc_options='version=GFC-2018-v1.6'),
    'hansen_latest':           dict(tcc_source='hansen', tcc_year='2026', tcc_options='version=GFC-2025-v1.13'),
    'meta_chm_v2_2m':          dict(tcc_source='meta_chm', tcc_year='2020', tcc_options='version=v2;ht_min=2'),
    'my_rasters':              dict(tcc_source='path', tcc_year='2020', tcc_paths='s3://<bucket>/<my_tcc>.vrt',
                                    tcc_options='resampling=average', out_name='mytcc2020'),
}
RUN = None        # set to one of RUNS
assert RUN in RUNS, f'set RUN to one of {list(RUNS)}'
params = dict(COMMON, **RUNS[RUN])
nn = lambda v: None if v in ('', 'none', None) else v
LABEL = ts.tcc_label(params['tcc_source'], params['tcc_year'], nn(params['tcc_options']), nn(params['out_name']))
IDENTIFIER = f'tte_map_{LABEL}'
print(ts.tcc_description(params['tcc_source'], params['tcc_year'], nn(params['tcc_options'])))
print('label:', LABEL, '| job tag:', IDENTIFIER)

tiles = gpd.read_file(os.path.expanduser('~/my-public-bucket/databank/tte_tiles.gpkg')).to_crs(4326)
print(len(tiles), 'tiles in the TTE tile index')


def _json(r):
    try:
        return r.json()
    except Exception:
        return {'http_status': getattr(r, 'status_code', None), 'text': getattr(r, 'text', str(r))[:300]}


import re, shlex

SUBMIT_LOG = os.path.expanduser('~/tte_dps_submissions')    # one CSV per job tag: every job submitted from here
OUT_ROOT = OUT_GLOB                     # this algorithm version's output folders (any deployed-name suffix)


def submitted(tag=IDENTIFIER):
    # jobs submitted from this notebook with this job tag
    f = os.path.join(SUBMIT_LOG, f'{tag}.csv')
    return pd.read_csv(f) if os.path.exists(f) else pd.DataFrame(columns=['tile_num', 'job_id', 'submit_time'])


def _log(rows, tag):
    os.makedirs(SUBMIT_LOG, exist_ok=True)
    df = pd.concat([submitted(tag), pd.DataFrame(rows)[['tile_num', 'job_id', 'submit_time']]], ignore_index=True)
    df.to_csv(os.path.join(SUBMIT_LOG, f'{tag}.csv'), index=False)


def _job_label(txt):
    # the run label of a job, from the 'python ... tte_map.py ...' line run.sh writes to _stdout (set -x)
    for line in txt.splitlines():
        if line.startswith('+ python') and 'tte_map.py' in line:
            tok = shlex.split(line[2:])
            arg = lambda k: tok[tok.index(k) + 1] if k in tok and tok.index(k) + 1 < len(tok) else None
            try:
                return ts.tcc_label(arg('--tcc_source'), arg('--tcc_year'), nn(arg('--tcc_options')), nn(arg('--out_name')))
            except Exception:
                return None
    return None


def tiles_done(label=LABEL):
    # {tile_num: reason} for tiles already made for this run label:
    #   'published'      STAC item with tte:tcc_label == label in the DPS STAC catalog
    #   'output file'    tte_<label>_<tile>.tif in dps_output (published or not yet)
    #   'outside domain' a job for this label ran and found no TTE domain (it writes no file)
    done = {}
    try:
        if not COLLECTION:
            raise RuntimeError('no STAC collection yet')
        from pystac_client import Client
        search = Client.open('https://dps-stac.maap-project.org').search(
            collections=[COLLECTION], filter={'op': '=', 'args': [{'property': 'tte:tcc_label'}, label]},
            filter_lang='cql2-json')
        for it in search.items():
            done[int(it.properties['tte:tile_num'])] = 'published'
    except Exception as e:
        print('STAC check skipped:', str(e)[:120])
    for f in glob.glob(f'{OUT_ROOT}/**/tte_{label}_*.tif', recursive=True):
        done.setdefault(int(os.path.basename(f)[:-4].rsplit('_', 1)[-1]), 'output file')
    for f in glob.glob(f'{OUT_ROOT}/**/_stdout*', recursive=True):
        try:
            txt = open(f, errors='ignore').read()
        except OSError:
            continue
        if 'outside the TTE domain' in txt and _job_label(txt) == label:
            for tn in re.findall(r'tile (\d+): bounds[^\n]*\n\s*domain pixels: 0\b', txt):
                done.setdefault(int(tn), 'outside domain')
    return done


def in_flight(tag=IDENTIFIER, exclude=()):
    # tiles with a job from the submission log that is still accepted or running
    s = submitted(tag)
    s = s[s.job_id.notna() & ~s.tile_num.isin(list(exclude))]
    busy = set()
    for tn, jid in zip(s.tile_num, s.job_id):
        st = str(_json(maap.get_job_status(jid)).get('status', '')).lower()
        if st in ('accepted', 'running', 'queued', 'deduped'):
            busy.add(int(tn))
    return busy


def todo(tile_list, label=LABEL, tag=IDENTIFIER, check_running=True):
    # the tiles of tile_list not done yet and without a job still running
    done = tiles_done(label)
    tile_list = list(dict.fromkeys(int(t) for t in tile_list))          # unique, in order
    rest = [t for t in tile_list if t not in done]
    busy = in_flight(tag, exclude=done) if check_running else set()
    rest = [t for t in rest if t not in busy]
    reasons = pd.Series([done[int(t)] for t in tile_list if int(t) in done], dtype=object).value_counts().to_dict()
    print(f'{label}: {len(tile_list)} tiles; done {reasons or 0}; queued/running {len(busy & set(map(int, tile_list)))}; '
          f'to submit {len(rest)}')
    return rest


def submit(tile_list, params, tag=IDENTIFIER, skip_done=True):
    # one OGC job per tile; inputs by name (run.sh reads --name value).
    # skip_done: leave out tiles already done for this run or with a job still running (see todo)
    if skip_done:
        tile_list = todo(tile_list, tag=tag)
    rows = []
    for tn in tile_list:
        r = maap.submit_job(PROCESS_ID, inputs=dict(params, tile_num=str(tn)), queue=QUEUE, tag=tag)
        j = _json(r)
        rows.append(dict(tile_num=tn, job_id=j.get('jobID') or j.get('id'), status=j.get('status'),
                         submit_time=datetime.datetime.now().isoformat(timespec='seconds'), response=j))
    if rows:
        _log(rows, tag)
    return pd.DataFrame(rows, columns=['tile_num', 'job_id', 'status', 'submit_time', 'response'])


def job_status(jobs):
    jobs = jobs.copy()
    jobs['status'] = [_json(maap.get_job_status(j)).get('status') for j in jobs.job_id]
    return jobs

## 3. Test one tile first

Check that the COG and the STAC files are written and that the item shows up in the DPS STAC catalog
before running anything at scale. Tile 4209 is the Kobuk-Noatak treeline in western Alaska.

`submit()` skips tiles already done for this run (`LABEL`): published in the DPS STAC catalog, with a COG
in `dps_output`, or found outside the TTE domain by an earlier job. It also skips tiles whose job (from the
submission log `~/tte_dps_submissions/<job tag>.csv`) is still queued or running. `todo(tiles)` shows
what would be submitted; `submit(..., skip_done=False)` submits regardless.

In [ ]:
TEST_TILE = 4209
# jobs_test = submit([TEST_TILE], params); jobs_test

In [ ]:
job_status(jobs_test)

In [ ]:
job_status(jobs_test)
_json(maap.get_job_result(jobs_test.job_id[0]))      # output links of a finished job

**3a. The job's output folder:** the COG, `catalog.json`, `tte_map/collection.json` and
`tte_map/<item id>/<item id>.json`. `pystac` re-validates the catalog as DPS will read it.

In [ ]:
import pystac
def job_dirs(tag=IDENTIFIER):
    # this algorithm version's job output folders whose job had this tag (read from the job's .met.json)
    out = []
    for met in glob.glob(f'{OUT_GLOB}/**/*.met.json', recursive=True):
        try:
            if json.load(open(met)).get('tag') == tag:
                out.append(os.path.dirname(met))
        except Exception:
            pass
    return sorted(out)          # the folders are named by date and time


dirs = job_dirs()
cats = [os.path.join(d, 'catalog.json') for d in dirs if os.path.exists(os.path.join(d, 'catalog.json'))]
print(len(dirs), 'job folder(s) with tag', IDENTIFIER, '| latest:', dirs[-1] if dirs else None)
if cats:
    job_dir = os.path.dirname(cats[-1])
    for f in sorted(glob.glob(f'{job_dir}/**/*', recursive=True)):
        if os.path.isfile(f):
            print(' ', os.path.relpath(f, job_dir), f'{os.path.getsize(f) / 1e6:.1f} MB')
    cat = pystac.Catalog.from_file(cats[-1])
    cat.validate_all()
    for it in cat.get_items(recursive=True):
        print('valid item:', it.id, '->', it.assets['tte'].get_absolute_href())
        print({k: v for k, v in it.properties.items() if k.startswith('tte:') and not isinstance(v, dict)})

**3b. Published in the DPS STAC catalog?** Ingestion is asynchronous; give it a few minutes after the
job finishes. Browser: `https://dps-stac-browser.maap-project.org` (your catalog appears there after the
first job that writes STAC).

In [ ]:
from pystac_client import Client
COLLECTION = COLLECTION or find_collection()
stac = Client.open('https://dps-stac.maap-project.org')
search = stac.search(collections=[COLLECTION],
                     filter={'op': '=', 'args': [{'property': 'tte:tile_num'}, TEST_TILE]}, filter_lang='cql2-json')
found = list(search.items())
print(len(found), 'item(s) for tile', TEST_TILE, 'in', COLLECTION)
for it in found:
    print(it.id, '| tag:', it.properties.get('maap-dps:tag'), '| created:', it.properties.get('created'))
    print('  asset:', it.assets['tte'].href)
print(f'https://dps-stac-browser.maap-project.org/collections/{COLLECTION}')

## 4. Review set: western Alaska and the Brooks Range

The tiles of the TTE domain in two boxes, for `notebooks/TTE_review_map.ipynb`. Tiles outside the TTE
domain exit without output.

In [ ]:
REVIEW_BOXES = {'western Alaska': (-166.0, 63.5, -155.0, 69.0), 'Brooks Range': (-155.0, 66.5, -141.0, 69.0)}
review = pd.concat([tiles[tiles.intersection(box(*b)).area > 0.01].assign(region=name) for name, b in REVIEW_BOXES.items()])
review = review.drop_duplicates('tile_num')
REVIEW_TILES = review.tile_num.tolist()
print(review.groupby('region').size().to_dict(), '->', len(REVIEW_TILES), 'tiles')
review.plot(column='region', legend=True, figsize=(8, 4), edgecolor='k', lw=0.3);

In [ ]:
REVIEW_TILES

In [ ]:
# jobs_review = submit(REVIEW_TILES, params)
# jobs_review.to_csv(os.path.expanduser(f'~/my-public-bucket/dps_submission_results/DPS_{ALGO_ID}_{IDENTIFIER}_review_{datetime.date.today()}.csv'), index=False)
# job_status(jobs_review).status.value_counts()

## 5. Compare with the GEE product (optional)
`validation/validate_tte_vs_gee.py` resamples a Python tile onto the matching GEE tile grid and reports agreement.

In [ ]:
# !python ~/code/tte/validation/validate_tte_vs_gee.py --tile_num 4209 --py_dir <job folder with the tif>

## 6. All tiles
Submit in batches. Hansen is ~1 min per tile on the 8 GB worker; Meta CHM 15-30 min.

In [ ]:
ALL_TILES = tiles.tile_num.tolist()
# jobs_all = submit(ALL_TILES, params)
# jobs_all.to_csv(os.path.expanduser(f'~/my-public-bucket/dps_submission_results/DPS_{ALGO_ID}_{IDENTIFIER}_{datetime.date.today()}.csv'), index=False)

## 7. Reference: publish the original c2010 map in the DPS STAC (once)

Your GEE map `projects/foreststructure/TTE_DATA_v201902v2`, downloaded as 4,809 one-degree COGs, is not in
the DPS STAC, so TiTiler can only map its 1 km overview (`lib/tte_overview.py`), not the 30 m tiles. MAAP's **DPS STAC Item Generator** (OGC process
`generate_stac_items`) lists a prefix and writes one STAC item per COG, pointing at the files where they
are (nothing is copied). Its items go to a collection named after MAAP's deployment
(`montesano__generate_stac_items_<n>__<version>`), shared with anything else you import, so the job tag
picks out these items: `notebooks/TTE_review_map.ipynb` finds them with `tr.find_reference()`.
Reading 4,809 file headers takes a while; submit once.

In [ ]:
REF_SOURCE = 's3://maap-ops-workspace/montesano/dps_output/run_download_gee_tiles/download_gee_tiles_v1/tte_montesano_2010_30m/'
REF_TAG = 'tte_data_v201902v2_reference'          # the review notebook looks for this job tag

gen = _json(maap.list_algorithms(algorithm_name='generate_stac_items', algorithm_version='0.3.2'))
gen = gen.get('processes', gen) if isinstance(gen, dict) else gen
GEN_PROCESS_ID = gen[0].get('processID') if gen else None
print('DPS STAC Item Generator processID:', GEN_PROCESS_ID)    # 134 (v0.3.2) when this was written

# r = maap.submit_job(GEN_PROCESS_ID, inputs={'source': REF_SOURCE, 'region': 'us-west-2', 'include_extensions': '.tif',
#                                             'exclude_extensions': '', 'config_json': ''},
#                     queue='maap-dps-worker-8gb', tag=REF_TAG)
# ref_job = _json(r); ref_job

In [ ]:
# _json(maap.get_job_status(ref_job.get('jobID') or ref_job.get('id')))